# 03 — Job Recommender (unsupervised)
TF-IDF the job corpus + resume → cosine similarity → top-N matching jobs.

In [3]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Locate the project folder
PROJECT_DIR = Path.cwd().parent

# Load the processed job dataset
jobs = pd.read_csv(
    PROJECT_DIR / "data" / "processed" / "jobs_clean.csv",
    usecols=lambda column: column in ["title", "company", "text"]
)

# Remove rows without job descriptions
jobs = jobs.dropna(subset=["text"]).reset_index(drop=True)

# Display dataset information
print("Job dataset loaded successfully!")
print("Dataset shape:", jobs.shape)
print("\nAvailable columns:")
print(jobs.columns.tolist())
print("\nFirst five jobs:")
display(jobs.head())


Job dataset loaded successfully!
Dataset shape: (21346, 3)

Available columns:
['title', 'company', 'text']

First five jobs:


,title,company,text
0,Walkin Data Entry Operator (night Shift),MM Media Pvt Ltd,walkin data entry operator night shift ites jo...
1,Work Based Onhome Based Part Time.,find live infotech,work based onhome based part time. marketing j...
2,Pl/sql Developer - SQL,Softtech Career Infosystem Pvt. Ltd,pl sql developer sql it software application p...
3,Manager/ad/partner - Indirect Tax - CA,Onboard HRServices LLP,manager ad partner indirect tax ca accounts jo...
4,JAVA Technical Lead (6-8 yrs) -,Spire Technologies and Solutions Pvt. Ltd.,java technical lead 6 8 yrs it software applic...


In [4]:
# Convert job descriptions into TF-IDF features
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english"
)

job_matrix = tfidf.fit_transform(jobs["text"].astype(str))

print("Job TF-IDF matrix created successfully!")
print("Matrix shape:", job_matrix.shape)
print("Number of jobs:", len(jobs))
print("Number of TF-IDF features:", len(tfidf.get_feature_names_out()))

Job TF-IDF matrix created successfully!
Matrix shape: (21346, 10000)
Number of jobs: 21346
Number of TF-IDF features: 10000


In [5]:
def recommend_jobs(resume_text, top_n=5):
    """
    Recommend jobs based on similarity to resume text.
    """
    if not isinstance(resume_text, str) or not resume_text.strip():
        print("Please provide valid resume text.")
        return pd.DataFrame()

    # Convert resume text using the same TF-IDF model
    resume_vector = tfidf.transform([resume_text])

    # Calculate similarity between the resume and all jobs
    similarity_scores = cosine_similarity(
        resume_vector, job_matrix
    ).flatten()

    # Select the top matching jobs
    top_indices = similarity_scores.argsort()[-top_n:][::-1]

    recommendations = jobs.iloc[top_indices][
        ["title", "company"]
    ].copy()

    recommendations["similarity_score"] = (
        similarity_scores[top_indices] * 100
    ).round(2)

    return recommendations.reset_index(drop=True)


print("Job recommendation function created successfully!")

Job recommendation function created successfully!


In [6]:
# Sample resume for testing
sample_resume = """
Python developer with skills in Python, SQL, data analysis,
machine learning, pandas, NumPy, and data visualization.
Experience in building machine learning projects and
working with databases.
"""

# Get the top 5 matching jobs
recommendations = recommend_jobs(sample_resume, top_n=5)

print("TOP 5 RECOMMENDED JOBS")
display(recommendations)

TOP 5 RECOMMENDED JOBS


,title,company,similarity_score
0,Software Engineer - Machine Learning,QUORA,51.64
1,Sr Scientist/ NLP/ Machine Learning/ Data Mini...,Career Maker,48.92
2,Machine Learning Scientist,Career Maker,48.90
3,Software Engineer,Startup - CloudInfra,47.85
4,Technical Lead - Python Developer - SQL,Premium hiring for Samaksh Consulting,46.64
